### **1. Import and Clean Dataset**

In [ ]:
import kagglehub
import pandas as pd
import os

# Download latest version
path = kagglehub.dataset_download("jordankrishnayah/45m-headlines-from-2007-2022-10-largest-sites")

print("Path to dataset files:", path)

# Construct the full path to the CSV file
csv_path = os.path.join(path, 'headlines.csv')

# save the data
data = pd.read_csv(csv_path)
display(data.head())

100%|██████████| 209M/209M [00:02<00:00, 76.5MB/s]

Extracting files...


Path to dataset files: /root/.cache/kagglehub/datasets/jordankrishnayah/45m-headlines-from-2007-2022-10-largest-sites/versions/2


,Date,Publication,Headline,URL
0,20070101,New York Times,Rush to Hang Hussein Was Questioned,http://www.nytimes.com/2007/01/01/world/middle...
1,20070101,New York Times,"News Analysis: For Sunnis, Dictators End Is O...",http://www.nytimes.com/2007/01/01/world/middle...
2,20070101,New York Times,Hard Choices Over Video,http://www.nytimes.com/2007/01/01/world/middle...
3,20070101,New York Times,States Take Lead on Ethics Rules for Lawmakers,http://www.nytimes.com/2007/01/01/us/01ethics....
4,20070101,New York Times,"Spitzer Arrives With Mandate, but Faces Challe...",http://www.nytimes.com/2007/01/01/nyregion/01e...


In [ ]:
cnn_headlines = data[data['Publication'] == 'CNN']

In [ ]:
# keep only the last instance of a URL
data = data.drop_duplicates(subset=['URL'], keep='last')
len(data)

2412320

In [ ]:
# remove duplicate headliens
unique_headlines = data.drop_duplicates(subset=['Headline'])
len(unique_headlines)

2177285

In [ ]:
data[data['Headline'] == '1 video']

,Date,Publication,Headline,URL
1909405,20140523,Daily Mail,1 video,http://www.dailymail.co.uk/travel/article-2637...
1910282,20140524,Daily Mail,1 video,http://www.dailymail.co.uk/sciencetech/article...
1911153,20140525,Daily Mail,1 video,http://www.dailymail.co.uk/news/article-263825...
1912031,20140526,Daily Mail,1 video,http://www.dailymail.co.uk/news/article-263972...
1912903,20140527,Daily Mail,1 video,http://www.dailymail.co.uk/sciencetech/article...
...,...,...,...,...
4401114,20221227,Daily Mail,1 video,https://www.dailymail.co.uk/news/article-11577...
4401950,20221228,Daily Mail,1 video,https://www.dailymail.co.uk/femail/article-115...
4402743,20221229,Daily Mail,1 video,https://www.dailymail.co.uk/news/article-11581...
4403585,20221230,Daily Mail,1 video,https://www.dailymail.co.uk/news/article-11586...


In [ ]:
# look at headlines with two or more instances
instances = data.value_counts('Headline')
len(instances[instances > 1])

unique_instances = instances[instances < 2]

In [ ]:
# remove duplicate headlines
data[data['Headline'].duplicated(keep=False)]

# remove NAN values in Headline
data = data.dropna(subset=['Headline'])

In [ ]:
# retrieve rows where the Headline is only one word
#data[data['Headline'].str.split().str.len() < 4]

# remove rows where the headline is 3 or less words
data = data[data['Headline'].str.split().str.len() > 3]

In [ ]:
# for each row, take the headline and break it into words
# then, pass these words into a feedforward network
# Input Parameter: Bag of Words
# output parameter: Headline

# STEP 1: CLEAN DOCUMENT TEXT to remove unusual characters
headlines = data['Headline'].str.lower()
headlines = headlines.str.replace('-', ' ', regex=False).str.replace("'s", "", regex=False).str.replace('/', ' ', regex=False).str.replace('ï¿½', '', regex=False)

# Create a translation table to remove specified characters
translator = str.maketrans('', '', ',".?!()[]:;$£%*')
headlines = headlines.str.translate(translator)

# STEP 2: SPLIT AND PROCESS ALL HEADLINES AT ONCE
# Split all headlines into words
all_words = headlines.str.split().explode()

# Remove apostrophes if at beginning or end of word
all_words = all_words[~(all_words.str.startswith("'") | all_words.str.endswith("'"))]

# STEP 3: REMOVE NUMBERS - filter out words containing digits
all_words = all_words[~all_words.str.contains(r'\d', regex=True, na=False)]

# STEP 4: GET UNIQUE WORDS - using set is much faster than list with checking
words_list = list(all_words.unique())
# add the dictionary to the list
#doc_dictionary_list.append(dictionary)

# define the index for each word
#word_to_index = {word: i for i, word in enumerate(all_words)}

# for all given words
#for word in all_words:
    #word_idx = word_to_index[word]

### **2. Create and Run Feedforward Network**

In [ ]:
import numpy as np
import tensorflow as tf
from tensorflow.keras import layers, models, utils
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
import pandas as pd

# Assuming you have:
# - headlines: cleaned text data
# - data['Publisher']: the target labels (10 news companies)

# Step 1: Prepare labels (10 news companies)
# Assuming your data has a 'Publisher' column with company names
labels = data['Publication']

# Create a tokenizer with your specific vocabulary
tokenizer = Tokenizer()
# Manually create word_index from your words_list
tokenizer.word_index = {word: idx+1 for idx, word in enumerate(words_list)}
tokenizer.index_word = {idx+1: word for idx, word in enumerate(words_list)}

# Get vocabulary size
vocab_size = len(tokenizer.word_index) + 1  # +1 for padding/index 0
print(f"Vocabulary size: {vocab_size}")

# Step 3: Convert headlines to sequences
sequences = tokenizer.texts_to_sequences(headlines)

# Step 4: Pad sequences to same length
max_length = max(len(seq) for seq in sequences)
# Or set a fixed max_length (e.g., 20-30 words typical for headlines)
max_length = min(30, max(len(seq) for seq in sequences))  # Cap at 30 words
padded_sequences = pad_sequences(sequences, maxlen=max_length, padding='post')

print(f"Sequence length: {max_length}")
print(f"Total samples: {len(padded_sequences)}")

# Step 5: Encode labels (10 news companies)
label_encoder = LabelEncoder()
encoded_labels = label_encoder.fit_transform(labels)
num_classes = len(label_encoder.classes_)

print(f"Classes: {label_encoder.classes_}")
print(f"Number of classes: {num_classes}")

# Convert labels to one-hot encoding for categorical crossentropy
one_hot_labels = utils.to_categorical(encoded_labels, num_classes=num_classes)

# Step 6: Split data
X_train, X_test, y_train, y_test = train_test_split(
    padded_sequences, one_hot_labels,
    test_size=0.2, random_state=42,
    stratify=encoded_labels  # Maintain class distribution
)

Vocabulary size: 242860
Sequence length: 30
Total samples: 2136329
Classes: ['BBC' 'CNBC' 'CNN' 'Daily Mail' 'FOX' 'New York Post' 'New York Times'
 'The Guardian' 'USA Today' 'Washington Post']
Number of classes: 10


## **CNN Network**

In [ ]:
# Step 7: Build feedforward neural network
model = models.Sequential([
    # Option A: Embedding + Flatten (Recommended for text)
    layers.Embedding(
        input_dim=vocab_size,
        output_dim=128,  # Embedding dimension - each word becomes 128-dim vector
        mask_zero=True  # Ignore padding
    ),
    layers.Flatten(),

    # Or Option B: Dense layers without embedding (if you want one-hot like approach)
    # layers.Input(shape=(max_length,)),
    # layers.Dense(128, activation='relu'),  # Would need custom one-hot encoding

    # Hidden layers
    layers.Dense(256, activation='relu'),
    layers.Dropout(0.4),
    layers.Dense(128, activation='relu'),
    layers.Dropout(0.3),
    layers.Dense(64, activation='relu'),
    layers.Dropout(0.2),

    # Output layer - 10 classes for 10 news companies
    layers.Dense(num_classes, activation='softmax')
])

# Step 8: Compile model
model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.001),
    loss='categorical_crossentropy',  # For multi-class classification
    metrics=['accuracy',
             tf.keras.metrics.Precision(name='precision'),
             tf.keras.metrics.Recall(name='recall')]
)

# Display model architecture
model.build(input_shape=(None, max_length))
model.summary()

# Step 9: Add callbacks for better training
callbacks = [
    tf.keras.callbacks.EarlyStopping(
        monitor='val_loss',
        patience=5,
        restore_best_weights=True
    ),
    tf.keras.callbacks.ReduceLROnPlateau(
        monitor='val_loss',
        factor=0.5,
        patience=3,
        min_lr=1e-6
    ),
    tf.keras.callbacks.ModelCheckpoint(
        'my_model.keras',
        monitor='val_accuracy',
        save_best_only=True
    )
]

# Step 10: Train the model
history = model.fit(
    X_train, y_train,
    validation_split=0.2,  # Use 20% of training as validation
    epochs=15,
    batch_size=1028,
    callbacks=callbacks,
    verbose=1
)

/usr/local/lib/python3.12/dist-packages/keras/src/layers/layer.py:965: UserWarning: Layer 'flatten_1' (of type Flatten) was passed an input with a mask attached to it. However, this layer does not support masking and will therefore destroy the mask information. Downstream layers will not see the mask.
  warnings.warn(


Model: "sequential_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding_1 (Embedding)         │ (None, 30, 128)        │    31,086,080 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten_1 (Flatten)             │ (None, 3840)           │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_4 (Dense)                 │ (None, 256)            │       983,296 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_3 (Dropout)             │ (None, 256)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_5 (Dense)                 │ (None, 128)            │        32,896 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_4 (Dropout)             │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_6 (Dense)                 │ (None, 64)             │         8,256 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_5 (Dropout)             │ (None, 64)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_7 (Dense)                 │ (None, 10)             │           650 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 32,111,178 (122.49 MB)

 Trainable params: 32,111,178 (122.49 MB)

 Non-trainable params: 0 (0.00 B)

Epoch 1/15
1331/1331 ━━━━━━━━━━━━━━━━━━━━ 37s 24ms/step - accuracy: 0.4134 - loss: 1.6403 - precision: 0.7866 - recall: 0.2169 - val_accuracy: 0.5856 - val_loss: 1.2025 - val_precision: 0.8009 - val_recall: 0.4133 - learning_rate: 0.0010
Epoch 2/15
1331/1331 ━━━━━━━━━━━━━━━━━━━━ 24s 18ms/step - accuracy: 0.6232 - loss: 1.1292 - precision: 0.8065 - recall: 0.4680 - val_accuracy: 0.5931 - val_loss: 1.1829 - val_precision: 0.7661 - val_recall: 0.4595 - learning_rate: 0.0010
Epoch 3/15
1331/1331 ━━━━━━━━━━━━━━━━━━━━ 18s 14ms/step - accuracy: 0.6845 - loss: 0.9572 - precision: 0.8328 - recall: 0.5624 - val_accuracy: 0.5905 - val_loss: 1.2206 - val_precision: 0.7323 - val_recall: 0.4811 - learning_rate: 0.0010
Epoch 4/15
 140/1331 ━━━━━━━━━━━━━━━━━━━━ 20s 18ms/step - accuracy: 0.7363 - loss: 0.8071 - precision: 0.8617 - recall: 0.6365

KeyboardInterrupt: 

In [ ]:
# Step 7: Build feedforward neural network
lstm_model = models.Sequential([
    # Option A: Embedding + Flatten (Recommended for text)
    layers.Embedding(
        input_dim=vocab_size,
        output_dim=128,  # Embedding dimension - each word becomes 128-dim vector
        mask_zero=False  # Factors in padding
    ),
    layers.LSTM(64, return_sequences=False),

    # Hidden layers
    layers.Dense(256, activation='relu'),
    layers.Dropout(0.4),
    layers.Dense(128, activation='relu'),
    layers.Dropout(0.3),
    layers.Dense(64, activation='relu'),
    layers.Dropout(0.2),

    # Output layer - 10 classes for 10 news companies
    layers.Dense(num_classes, activation='softmax')
])

# Step 8: Compile model
lstm_model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.001),
    loss='categorical_crossentropy',  # For multi-class classification
    metrics=['accuracy',
             tf.keras.metrics.Precision(name='precision'),
             tf.keras.metrics.Recall(name='recall')]
)

# Display model architecture
lstm_model.build(input_shape=(None, max_length))
lstm_model.summary()

# Step 9: Add callbacks for better training
callbacks = [
    tf.keras.callbacks.EarlyStopping(
        monitor='val_loss',
        patience=5,
        restore_best_weights=True
    ),
    tf.keras.callbacks.ReduceLROnPlateau(
        monitor='val_loss',
        factor=0.5,
        patience=3,
        min_lr=1e-6
    ),
    tf.keras.callbacks.ModelCheckpoint(
        'my_model.keras',
        monitor='val_accuracy',
        save_best_only=True
    )
]

# Step 10: Train the model
history = lstm_model.fit(
    X_train, y_train,
    validation_split=0.2,  # Use 20% of training as validation
    epochs=10,
    batch_size=4112,
    callbacks=callbacks,
    verbose=1
)

Model: "sequential_3"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding_3 (Embedding)         │ (None, 30, 128)        │    31,086,080 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm_1 (LSTM)                   │ (None, 64)             │        49,408 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_12 (Dense)                │ (None, 256)            │        16,640 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_9 (Dropout)             │ (None, 256)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_13 (Dense)                │ (None, 128)            │        32,896 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_10 (Dropout)            │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_14 (Dense)                │ (None, 64)             │         8,256 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_11 (Dropout)            │ (None, 64)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_15 (Dense)                │ (None, 10)             │           650 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 31,193,930 (119.00 MB)

 Trainable params: 31,193,930 (119.00 MB)

 Non-trainable params: 0 (0.00 B)

Epoch 1/10
333/333 ━━━━━━━━━━━━━━━━━━━━ 28s 77ms/step - accuracy: 0.2449 - loss: 2.0084 - precision: 0.7841 - recall: 0.0764 - val_accuracy: 0.4271 - val_loss: 1.5768 - val_precision: 0.8346 - val_recall: 0.1922 - learning_rate: 0.0010
Epoch 2/10
333/333 ━━━━━━━━━━━━━━━━━━━━ 34s 101ms/step - accuracy: 0.4577 - loss: 1.5284 - precision: 0.7865 - recall: 0.2409 - val_accuracy: 0.5161 - val_loss: 1.3865 - val_precision: 0.7977 - val_recall: 0.3171 - learning_rate: 0.0010
Epoch 3/10
333/333 ━━━━━━━━━━━━━━━━━━━━ 34s 101ms/step - accuracy: 0.5419 - loss: 1.3366 - precision: 0.7964 - recall: 0.3535 - val_accuracy: 0.5411 - val_loss: 1.3207 - val_precision: 0.7799 - val_recall: 0.3716 - learning_rate: 0.0010
Epoch 4/10
333/333 ━━━━━━━━━━━━━━━━━━━━ 38s 116ms/step - accuracy: 0.5851 - loss: 1.2241 - precision: 0.8145 - recall: 0.4132 - val_accuracy: 0.5582 - val_loss: 1.2852 - val_precision: 0.7700 - val_recall: 0.4043 - learning_rate: 0.0010
Epoch 5/10
333/333 ━━━━━━━━━━━━━━━━━━━━ 38s 113ms/ste

In [ ]:
def analyze_publisher_style_with_frequency(publisher_name, model, tokenizer,
                                          X_train_sequences, y_train,
                                          label_encoder, top_n=15):
    """
    Analyze publisher style with frequency context
    Returns words with their: weight, frequency, and distinctiveness
    """
    publisher_idx = list(label_encoder.classes_).index(publisher_name)

    # Get publisher weights from final dense layer
    dense_layer = model.layers[-1]
    publisher_weights = dense_layer.get_weights()[0][:, publisher_idx]

    # Get indices sorted by weight (most to least important)
    important_indices = publisher_weights.argsort()[::-1][:50]  # Get top 50

    # Count word frequencies for THIS publisher
    publisher_mask = (np.argmax(y_train, axis=1) == publisher_idx)
    publisher_sequences = X_train_sequences[publisher_mask]

    # Count frequencies
    word_freqs = {}
    for seq in publisher_sequences:
        for idx in seq:
            if idx != 0:  # Skip padding
                word_freqs[idx] = word_freqs.get(idx, 0) + 1

    # Also count frequencies in ALL publishers for comparison
    total_word_freqs = {}
    for seq in X_train_sequences:
        for idx in seq:
            if idx != 0:
                total_word_freqs[idx] = total_word_freqs.get(idx, 0) + 1

    # Analyze each important word
    results = []
    for idx in important_indices[:30]:  # Analyze top 30
        if idx in tokenizer.index_word and idx != 0:
            word = tokenizer.index_word[idx]
            weight = publisher_weights[idx]

            # Get frequencies
            freq_in_publisher = word_freqs.get(idx, 0)
            freq_total = total_word_freqs.get(idx, 0)

            # Calculate percentage of usage in this publisher
            if freq_total > 0:
                percentage_in_publisher = (freq_in_publisher / freq_total) * 100
            else:
                percentage_in_publisher = 0

            # Calculate frequency rank in this publisher
            all_freqs = list(word_freqs.values())
            if freq_in_publisher > 0:
                freq_percentile = np.percentile(all_freqs,
                    (freq_in_publisher / max(all_freqs)) * 100) if all_freqs else 0
            else:
                freq_percentile = 0

            results.append({
                'word': word,
                'weight': weight,
                'freq_in_publisher': freq_in_publisher,
                'freq_total': freq_total,
                'percentage_in_publisher': percentage_in_publisher,
                'is_frequent': freq_in_publisher > np.median(list(word_freqs.values())) if word_freqs else False,
                'is_distinctive': percentage_in_publisher > 20  # More than 20% of usage is from this publisher
            })

    # Sort by a combined score that considers both weight AND distinctiveness
    for r in results:
        # Combine weight (normalized) with distinctiveness score
        weight_score = r['weight']
        distinctiveness_score = r['percentage_in_publisher']
        frequency_score = min(r['freq_in_publisher'], 100) / 100  # Cap at 100

        # Combined score: weight matters, but so does being distinctive
        r['combined_score'] = (weight_score * 0.5 +
                              distinctiveness_score * 0.3 +
                              frequency_score * 0.2)

    # Sort by combined score
    results.sort(key=lambda x: x['combined_score'], reverse=True)

    # Categorize words
    categories = {
        'style_signatures': [],  # High weight + distinctive + somewhat frequent
        'common_words': [],      # High weight + very frequent but not distinctive
        'rare_indicators': [],   # High weight + rare but highly distinctive
        'ignored': []            # Low weight or not distinctive
    }

    for r in results[:top_n*2]:  # Analyze more than we need
        if r['weight'] > 0 and r['is_distinctive']:
            if r['freq_in_publisher'] > 10:  # Appears at least 10 times
                categories['style_signatures'].append(r)
            else:
                categories['rare_indicators'].append(r)
        elif r['freq_in_publisher'] > 50:  # Very frequent
            categories['common_words'].append(r)
        else:
            categories['ignored'].append(r)

    return results[:top_n], categories

# Run analysis for all publishers
for publisher in ["BBC", "CNBC", "CNN", "Daily Mail", "FOX", "New York Post", "New York Times", "The Guardian", "USA Today", "Washington Post"]:
    print(f"\n{'='*60}")
    print(f"Publisher: {publisher}")
    print('='*60)

    top_words, categories = analyze_publisher_style_with_frequency(
        publisher, model, tokenizer, X_train, y_train, label_encoder
    )

    print("\n🎯 STYLE SIGNATURE WORDS (most meaningful):")
    for r in categories['style_signatures'][:8]:
        print(f"  {r['word']:15} weight:{r['weight']:7.3f} freq:{r['freq_in_publisher']:3d} "
              f"distinct:{r['percentage_in_publisher']:5.1f}%")

    print("\n📊 COMMON WORDS (frequent but not distinctive):")
    for r in categories['common_words'][:5]:
        print(f"  {r['word']:15} freq:{r['freq_in_publisher']:3d}")

    print("\n🔍 RARE BUT DISTINCTIVE (unique markers):")
    for r in categories['rare_indicators'][:5]:
        print(f"  {r['word']:15} weight:{r['weight']:7.3f} freq:{r['freq_in_publisher']:2d}")

In [ ]:
def analyze_publisher_style(publisher_name, num_examples=5):
    """Analyze what makes a publisher's style unique"""
    publisher_idx = list(label_encoder.classes_).index(publisher_name)

    # Get model's feature importance (simplified)
    embedding_layer = model.layers[0]
    dense_layer = model.layers[-1]  # Output layer

    # Get weights for this publisher
    publisher_weights = dense_layer.get_weights()[0][:, publisher_idx]

    # Sort words by importance (simplified approach)
    important_indices = publisher_weights.argsort()[-10:][::-1]

    # Map back to words
    important_words = []
    for idx in important_indices:
        # Ensure idx is within the range of tokenizer.word_index values
        if idx in tokenizer.index_word:
            important_words.append(tokenizer.index_word[idx])

    return important_words

# run the analyze publishers styles function
for publisher in ["BBC", "CNBC", "CNN", "Daily Mail", "FOX", "New York Post", "New York Times", "The Guardian", "USA Today", "Washington Post"]:
    print(f"Publisher: {publisher}")
    print(analyze_publisher_style(publisher))
    print()

In [ ]:
# find headlines with 'team' or 'Team' in the headline
team_data = data[data['Headline'].str.lower().str.contains(' sworn ')]

# Get counts for each publisher (with 'team')
team_counts = team_data['Publication'].value_counts()

# Get total counts for each publisher (all headlines)
total_counts = data['Publication'].value_counts()

# Calculate percentages
percentage_by_publisher = (team_counts / total_counts * 100).sort_values(ascending=False)

print("Percentage of headlines containing 'team' by Publisher:")
print(percentage_by_publisher)

CNN Model

### **3. Evaluation & Analysis of Model**

In [ ]:
# Step 11: Evaluate on test set
test_loss, test_accuracy, test_precision, test_recall = model.evaluate(X_test, y_test, verbose=0)
print(f"\nTest Results:")
print(f"Accuracy: {test_accuracy:.4f}")
print(f"Precision: {test_precision:.4f}")
print(f"Recall: {test_recall:.4f}")

# Step 12: Make predictions
predictions = model.predict(X_test)
predicted_classes = np.argmax(predictions, axis=1)
true_classes = np.argmax(y_test, axis=1)

# Step 13: Create classification report
from sklearn.metrics import classification_report, confusion_matrix

print("\nClassification Report:")
print(classification_report(
    true_classes,
    predicted_classes,
    target_names=label_encoder.classes_
))

# Step 14: Save the model and tokenizer for later use
model.save('news_classifier.h5')

import pickle
with open('tokenizer.pkl', 'wb') as f:
    pickle.dump(tokenizer, f)

with open('label_encoder.pkl', 'wb') as f:
    pickle.dump(label_encoder, f)

In [ ]:
# Function to predict new headlines
def predict_headline(new_headline):
    """Predict the news company for a new headline"""
    # Clean the new headline using the same process
    cleaned = new_headline.lower()
    cleaned = (cleaned.replace('-', ' ')
                       .replace("'s", "")
                       .replace('/', ' ')
                       .replace('ï¿½', ''))
    translator = str.maketrans('', '', ',".?!()[]:;$£%*')
    cleaned = cleaned.translate(translator)

    # Tokenize and pad
    seq = tokenizer.texts_to_sequences([cleaned])
    padded = pad_sequences(seq, maxlen=max_length, padding='post')

    # Predict
    prediction = model.predict(padded, verbose=0)
    predicted_class = np.argmax(prediction[0])
    confidence = np.max(prediction[0])

    return label_encoder.inverse_transform([predicted_class])[0], confidence

# Test the function
test_headline = "No Christmas homecoming: Families mourn victims of Goa nightclub fire"
company, confidence = predict_headline(test_headline)
print(f"\nPrediction for '{test_headline}':")
print(f"Company: {company}, Confidence: {confidence:.2%}")

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import confusion_matrix, classification_report

# Assuming you have:
# predictions, predicted_classes, true_classes from your code
# label_encoder for decoding class names

# 1. Create confusion matrix
conf_matrix = confusion_matrix(true_classes, predicted_classes)

# 2. Get class names
class_names = label_encoder.classes_

# 3. Analyze most common misclassifications
print("ANALYSIS OF MISLABELED CLASSES")
print("=" * 60)

# Create DataFrame for easier analysis
conf_df = pd.DataFrame(conf_matrix,
                      index=[f"True_{name}" for name in class_names],
                      columns=[f"Pred_{name}" for name in class_names])

# Calculate metrics
total_samples = np.sum(conf_matrix)
correct_predictions = np.diag(conf_matrix)
incorrect_predictions = np.sum(conf_matrix, axis=1) - correct_predictions

# Create summary DataFrame
analysis_df = pd.DataFrame({
    'Class': class_names,
    'Total_Samples': np.sum(conf_matrix, axis=1),
    'Correct': correct_predictions,
    'Incorrect': incorrect_predictions,
    'Accuracy': correct_predictions / np.sum(conf_matrix, axis=1)
})

# Sort by most least to most accuracy percentage
analysis_df = analysis_df.sort_values('Accuracy', ascending=True)

print("\n1. Classes with most misclassifications (worst performers):")
print(analysis_df[['Class', 'Total_Samples', 'Incorrect', 'Accuracy']].head())

# 4. Identify specific confusion patterns
print("\n2. Most common confusion pairs:")
conf_pairs = []

# Get all off-diagonal elements (misclassifications)
for i in range(len(class_names)):
    for j in range(len(class_names)):
        if i != j and conf_matrix[i, j] > 0:
            conf_pairs.append({
                'True_Class': class_names[i],
                'Predicted_Class': class_names[j],
                'Count': conf_matrix[i, j],
                'Percentage': conf_matrix[i, j] / np.sum(conf_matrix[i]) * 100
            })

# Convert to DataFrame and sort
conf_pairs_df = pd.DataFrame(conf_pairs)
if not conf_pairs_df.empty:
    conf_pairs_df = conf_pairs_df.sort_values('Percentage', ascending=False)
    print(conf_pairs_df.head(10).to_string(index=False))
else:
    print("No misclassifications found (perfect model!)")

# 5. For each class, show what it's most commonly confused with
print("\n3. For each class, what it's most often confused with:")
for i, true_class in enumerate(class_names):
    row = conf_matrix[i]
    total = np.sum(row)
    if total > 0:
        # Get indices sorted by misclassification count (excluding diagonal)
        misclass_indices = np.argsort(row)[::-1]
        # Filter out self (diagonal) and get top 3
        top_misclass = []
        for idx in misclass_indices:
            if idx != i and row[idx] > 0:
                top_misclass.append((class_names[idx], row[idx], row[idx]/total*100))
                if len(top_misclass) >= 3:
                    break

        if top_misclass:
            print(f"\n{true_class} (Total: {total}, Correct: {row[i]}, Accuracy: {row[i]/total*100:.1f}%)")
            for pred_class, count, percent in top_misclass:
                print(f"  → Mislabeled as {pred_class}: {count} samples ({percent:.1f}%)")

# 6. Visualize confusion matrix
plt.figure(figsize=(12, 10))
sns.heatmap(conf_matrix,
           annot=True,
           fmt='d',
           cmap='Blues',
           xticklabels=class_names,
           yticklabels=class_names)
plt.title('Confusion Matrix', fontsize=16)
plt.xlabel('Predicted Label', fontsize=14)
plt.ylabel('True Label', fontsize=14)
plt.tight_layout()
plt.savefig('confusion_matrix.png', dpi=300, bbox_inches='tight')
plt.show()

# 7. Create normalized confusion matrix (by row)
plt.figure(figsize=(12, 10))
conf_matrix_norm = conf_matrix.astype('float') / conf_matrix.sum(axis=1)[:, np.newaxis]
conf_matrix_norm = np.nan_to_num(conf_matrix_norm)  # Handle division by zero

sns.heatmap(conf_matrix_norm,
           annot=True,
           fmt='.2f',
           cmap='Reds',
           xticklabels=class_names,
           yticklabels=class_names)
plt.title('Normalized Confusion Matrix (by True Label)', fontsize=16)
plt.xlabel('Predicted Label', fontsize=14)
plt.ylabel('True Label', fontsize=14)
plt.tight_layout()
plt.savefig('confusion_matrix_normalized.png', dpi=300, bbox_inches='tight')
plt.show()

# 8. Detailed error analysis for specific classes
print("\n4. Detailed error analysis for poorly performing classes:")
# Get classes with accuracy < threshold (e.g., 70%)
threshold = 0.7
poor_performers = analysis_df[analysis_df['Accuracy'] < threshold]

for _, row in poor_performers.iterrows():
    class_idx = np.where(class_names == row['Class'])[0][0]
    print(f"\n{'='*60}")
    print(f"ERROR ANALYSIS FOR: {row['Class']}")
    print(f"{'='*60}")
    print(f"Total samples: {row['Total_Samples']}")
    print(f"Correct: {row['Correct']}")
    print(f"Incorrect: {row['Incorrect']}")
    print(f"Accuracy: {row['Accuracy']*100:.1f}%")

    # Get distribution of predictions for this true class
    pred_distribution = conf_matrix[class_idx]
    print("\nPrediction distribution:")
    for j, pred_class in enumerate(class_names):
        if pred_distribution[j] > 0:
            percentage = pred_distribution[j] / row['Total_Samples'] * 100
            print(f"  {pred_class}: {pred_distribution[j]} ({percentage:.1f}%)")

# 9. Create misclassification matrix (off-diagonal only)
print("\n5. Misclassification matrix (only incorrect predictions):")
misclass_matrix = conf_matrix.copy()
np.fill_diagonal(misclass_matrix, 0)  # Zero out correct predictions

if misclass_matrix.sum() > 0:
    # Find the most common misclassification pair
    max_misclass = np.unravel_index(np.argmax(misclass_matrix), misclass_matrix.shape)
    true_max, pred_max = max_misclass
    print(f"\nMost common misclassification:")
    print(f"True: {class_names[true_max]} → Predicted: {class_names[pred_max]}")
    print(f"Count: {misclass_matrix[true_max, pred_max]}")

    # Plot misclassification heatmap
    plt.figure(figsize=(12, 10))
    sns.heatmap(misclass_matrix,
               annot=True,
               fmt='d',
               cmap='Oranges',
               xticklabels=class_names,
               yticklabels=class_names)
    plt.title('Misclassification Matrix (Correct Predictions Removed)', fontsize=16)
    plt.xlabel('Predicted Label', fontsize=14)
    plt.ylabel('True Label', fontsize=14)
    plt.tight_layout()
    plt.savefig('misclassification_matrix.png', dpi=300, bbox_inches='tight')
    plt.show()

# 10. Save detailed analysis to CSV
# Create detailed error analysis DataFrame
error_details = []

for i, true_class in enumerate(class_names):
    for j, pred_class in enumerate(class_names):
        if i != j and conf_matrix[i, j] > 0:
            error_details.append({
                'true_class': true_class,
                'predicted_class': pred_class,
                'count': conf_matrix[i, j],
                'percentage_of_true_class': conf_matrix[i, j] / np.sum(conf_matrix[i]) * 100,
                'percentage_of_all_errors': conf_matrix[i, j] / (total_samples - np.sum(np.diag(conf_matrix))) * 100
            })

error_df = pd.DataFrame(error_details)
if not error_df.empty:
    error_df = error_df.sort_values(['count', 'percentage_of_all_errors'], ascending=False)
    error_df.to_csv('error_analysis.csv', index=False)
    print(f"\nDetailed error analysis saved to 'error_analysis.csv'")

# 11. Summary statistics
print("\n" + "="*60)
print("SUMMARY STATISTICS")
print("="*60)
print(f"Total test samples: {total_samples}")
print(f"Total correct: {np.sum(correct_predictions)}")
print(f"Total incorrect: {np.sum(incorrect_predictions)}")
print(f"Overall accuracy: {np.sum(correct_predictions)/total_samples*100:.2f}%")

# Class-wise statistics
print("\nClass-wise statistics:")
for i, class_name in enumerate(class_names):
    total = np.sum(conf_matrix[i])
    if total > 0:
        accuracy = conf_matrix[i, i] / total * 100
        print(f"{class_name:20s}: {conf_matrix[i, i]:3d} / {total:3d} = {accuracy:5.1f}%")

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import confusion_matrix

# Create confusion matrix
conf_matrix = confusion_matrix(true_classes, predicted_classes)
class_names = label_encoder.classes_

# 1. Calculate confusion percentages (normalized by true class)
conf_percentages = conf_matrix.astype('float') / conf_matrix.sum(axis=1)[:, np.newaxis] * 100
conf_percentages = np.nan_to_num(conf_percentages)

# 2. Extract only misclassifications (off-diagonal)
misclass_data = []
for i in range(len(class_names)):
    for j in range(len(class_names)):
        if i != j and conf_percentages[i, j] > 0:
            misclass_data.append({
                'True Publisher': class_names[i],
                'Predicted Publisher': class_names[j],
                'Error Rate (%)': conf_percentages[i, j],
                'Count': conf_matrix[i, j]
            })

# Create DataFrame and sort by error rate
error_df = pd.DataFrame(misclass_data)
error_df = error_df.sort_values('Error Rate (%)', ascending=False)

# 3. Display top confusions
print("=" * 80)
print("TOP PUBLISHER CONFUSIONS (Most Common Mistakes)")
print("=" * 80)
print("\nTop 15 most common confusion patterns:\n")
print(error_df.head(15).to_string(index=False))

# 4. Create heatmap showing only significant confusions (> 5%)
significant_threshold = 5
conf_percentages_filtered = conf_percentages.copy()
conf_percentages_filtered[conf_percentages_filtered < significant_threshold] = 0
np.fill_diagonal(conf_percentages_filtered, 0)  # Remove correct predictions

# Create figure with multiple subplots
fig, axes = plt.subplots(2, 2, figsize=(18, 16))

# Plot 1: Full confusion matrix (percentages)
ax1 = axes[0, 0]
sns.heatmap(conf_percentages,
            annot=True,
            fmt='.1f',
            cmap='YlOrRd',
            xticklabels=class_names,
            yticklabels=class_names,
            cbar_kws={'label': 'Percentage (%)'},
            ax=ax1)
ax1.set_title('Complete Confusion Matrix (%)', fontsize=14, fontweight='bold')
ax1.set_xlabel('Predicted Publisher', fontsize=12)
ax1.set_ylabel('True Publisher', fontsize=12)

# Plot 2: Significant confusions only (> 5%)
ax2 = axes[0, 1]
sns.heatmap(conf_percentages_filtered,
            annot=True,
            fmt='.1f',
            cmap='Reds',
            xticklabels=class_names,
            yticklabels=class_names,
            cbar_kws={'label': 'Percentage (%)'},
            ax=ax2,
            vmin=0)
ax2.set_title(f'Significant Confusions (>{significant_threshold}% error rate)',
              fontsize=14, fontweight='bold')
ax2.set_xlabel('Predicted Publisher', fontsize=12)
ax2.set_ylabel('True Publisher', fontsize=12)

# Plot 3: Bar chart of top 10 confusion pairs
ax3 = axes[1, 0]
top_10 = error_df.head(10).copy()
top_10['Confusion Pair'] = top_10['True Publisher'] + ' → ' + top_10['Predicted Publisher']
ax3.barh(range(len(top_10)), top_10['Error Rate (%)'], color='coral')
ax3.set_yticks(range(len(top_10)))
ax3.set_yticklabels(top_10['Confusion Pair'], fontsize=10)
ax3.set_xlabel('Error Rate (%)', fontsize=12)
ax3.set_title('Top 10 Most Common Confusions', fontsize=14, fontweight='bold')
ax3.invert_yaxis()
for i, v in enumerate(top_10['Error Rate (%)']):
    ax3.text(v + 0.5, i, f'{v:.1f}%', va='center', fontsize=9)

# Plot 4: Publisher accuracy summary
ax4 = axes[1, 1]
accuracy_by_class = []
for i, class_name in enumerate(class_names):
    total = conf_matrix[i].sum()
    correct = conf_matrix[i, i]
    accuracy = (correct / total * 100) if total > 0 else 0
    accuracy_by_class.append({
        'Publisher': class_name,
        'Accuracy (%)': accuracy,
        'Total Samples': total
    })

acc_df = pd.DataFrame(accuracy_by_class).sort_values('Accuracy (%)')
colors = ['red' if x < 70 else 'orange' if x < 85 else 'green' for x in acc_df['Accuracy (%)']]
ax4.barh(range(len(acc_df)), acc_df['Accuracy (%)'], color=colors)
ax4.set_yticks(range(len(acc_df)))
ax4.set_yticklabels(acc_df['Publisher'], fontsize=10)
ax4.set_xlabel('Accuracy (%)', fontsize=12)
ax4.set_title('Publisher Classification Accuracy', fontsize=14, fontweight='bold')
ax4.axvline(x=85, color='gray', linestyle='--', alpha=0.5, label='85% threshold')
ax4.legend()
for i, (acc, count) in enumerate(zip(acc_df['Accuracy (%)'], acc_df['Total Samples'])):
    ax4.text(acc + 1, i, f'{acc:.1f}% (n={count})', va='center', fontsize=9)

plt.tight_layout()
plt.savefig('publisher_confusion_analysis.png', dpi=300, bbox_inches='tight')
plt.show()

# 5. Summary table by publisher
print("\n" + "=" * 80)
print("PUBLISHER PERFORMANCE SUMMARY")
print("=" * 80)
summary_data = []
for i, class_name in enumerate(class_names):
    total = conf_matrix[i].sum()
    correct = conf_matrix[i, i]
    incorrect = total - correct
    accuracy = (correct / total * 100) if total > 0 else 0

    # Find most common mistake
    row = conf_matrix[i].copy()
    row[i] = 0  # Exclude correct predictions
    if row.max() > 0:
        most_confused_idx = row.argmax()
        most_confused = class_names[most_confused_idx]
        confusion_pct = row[most_confused_idx] / total * 100
    else:
        most_confused = "N/A"
        confusion_pct = 0

    summary_data.append({
        'Publisher': class_name,
        'Total': total,
        'Correct': correct,
        'Wrong': incorrect,
        'Accuracy (%)': f'{accuracy:.1f}',
        'Most Confused With': most_confused,
        'Confusion Rate (%)': f'{confusion_pct:.1f}'
    })

summary_df = pd.DataFrame(summary_data)
summary_df = summary_df.sort_values('Accuracy (%)', ascending=False)
print("\n" + summary_df.to_string(index=False))

# 6. Identify publisher groups with mutual confusion
print("\n" + "=" * 80)
print("MUTUAL CONFUSION PAIRS (Publishers often confused with each other)")
print("=" * 80)
mutual_confusions = []
for i in range(len(class_names)):
    for j in range(i+1, len(class_names)):
        conf_i_to_j = conf_percentages[i, j]
        conf_j_to_i = conf_percentages[j, i]
        if conf_i_to_j > 3 and conf_j_to_i > 3:  # Both directions > 3%
            avg_confusion = (conf_i_to_j + conf_j_to_i) / 2
            mutual_confusions.append({
                'Publisher 1': class_names[i],
                'Publisher 2': class_names[j],
                f'{class_names[i]} → {class_names[j]} (%)': f'{conf_i_to_j:.1f}',
                f'{class_names[j]} → {class_names[i]} (%)': f'{conf_j_to_i:.1f}',
                'Avg Confusion (%)': f'{avg_confusion:.1f}'
            })

if mutual_confusions:
    mutual_df = pd.DataFrame(mutual_confusions)
    mutual_df = mutual_df.sort_values('Avg Confusion (%)', ascending=False)
    print("\n" + mutual_df.to_string(index=False))
else:
    print("\nNo significant mutual confusions found (threshold: 3%)")

print("\n" + "=" * 80)

In [ ]:
# Recreate the split exactly as before
original_indices = np.arange(len(headlines))

# You need the same encoded_labels used originally
# If you don't have it, recreate it:
encoded_labels = label_encoder.transform(labels)

# Recreate the split
train_idx, test_idx = train_test_split(
    original_indices,
    test_size=0.2,
    random_state=42,
    stratify=encoded_labels
)

# Now you have test_idx which matches your X_test order

In [ ]:
# X_test contains the indices of the test examples
# y_test contains whether the example is true or not
# predicted_classes contains the class predicted
# true_classes

# merge X_test, y_test, predicted_classes, and true_classes into one dataframe
# Now get test headlines
test_headlines = headlines.iloc[test_idx].reset_index(drop=True)

# Get predictions
predictions = model.predict(X_test)
predicted_classes_idx = np.argmax(predictions, axis=1)
true_classes_idx = np.argmax(y_test, axis=1)

# Create misclass_df
misclass_data = []
for i in range(len(X_test)):
    misclass_data.append({
        'headline': test_headlines.iloc[i],
        'true_class': label_encoder.inverse_transform([true_classes_idx[i]])[0],
        'predicted_class': label_encoder.inverse_transform([predicted_classes_idx[i]])[0],
        'confidence': np.max(predictions[i]),
        'is_correct': predicted_classes_idx[i] == true_classes_idx[i],
        'original_index': test_idx[i]
    })

misclass_df = pd.DataFrame(misclass_data)

# Filter to only incorrect predictions
incorrect_df = misclass_df[misclass_df['is_correct'] == False]
print(f"Found {len(incorrect_df)} misclassifications")

In [ ]:
incorrect_df

In [ ]:
# count the average number of words by Publication
for publication in data['Publication'].unique():
    publication_headlines = data[data['Publication'] == publication]
    num_words = publication_headlines['Headline'].str.split().apply(len)
    avg_words = num_words.mean()
    print(f"{publication}: {avg_words:.2f} words per headline")

In [ ]:
# data = data[data['Headline'].str.split().str.len() > 3]

# **DEMO**: Scraper of News Websites

### **CNN Scraper**

In [ ]:
#import scraping libraries
import requests
from bs4 import BeautifulSoup

# request to the CNN Website
url = "https://lite.cnn.com"

response = requests.get(url)

soup = BeautifulSoup(response.text, 'html.parser')

# retrieve all links
links = soup.find_all('a')

cnn_headlines = []

for link in links:
    # if href includes '/2025'
    if '/2025' in link.get('href', ''):
        # save headline to cnn_headlines
        cnn_headlines.append(link.text.strip())

#print(cnn_headlines)

cnn_headlines_df = pd.DataFrame(cnn_headlines, columns=['Headline'])

cnn_headlines_df['Predicted Publication'] = ''

# pass each headline into predict_headline
for headline in cnn_headlines:
    company, confidence = predict_headline(headline)
    # print company and confidence
    print(f"Prediction for '{headline}': {company}")
    # save headline with company into a dataframe
    cnn_headlines_df.loc[cnn_headlines_df['Headline'] == headline, 'Predicted Publication'] = company

# count the accuracy
accuracy = (cnn_headlines_df['Predicted Publication'] == 'CNN').mean()
print(f"CNN Accuracy: {accuracy*100:.2f}%")
print(len(cnn_headlines_df))

## **Daily Mail Scraper**

In [ ]:
#import scraping libraries
import requests
from bs4 import BeautifulSoup

# request to the daily_mail Website
url = "https://www.dailymail.co.uk/textbased/channel-561/index.html"

response = requests.get(url)

soup = BeautifulSoup(response.text, 'html.parser')

# retrieve all links
links = soup.find_all('a')

daily_mail_headlines = []

for link in links:
    headings = link.find_all('h3')
    for heading in headings:
        # retrieve the text
        headline = heading.text.strip()
        if headline:
            daily_mail_headlines.append(headline)

#print(daily_mail_headlines)

daily_mail_headlines_df = pd.DataFrame(daily_mail_headlines, columns=['Headline'])

daily_mail_headlines_df['Predicted Publication'] = ''

# pass each headline into predict_headline
for headline in daily_mail_headlines:
    company, confidence = predict_headline(headline)
    # print company and confidence
    print(f"Prediction for '{headline}': {company}")
    # save headline with company into a dataframe
    daily_mail_headlines_df.loc[daily_mail_headlines_df['Headline'] == headline, 'Predicted Publication'] = company

# count the accuracy
accuracy = (daily_mail_headlines_df['Predicted Publication'] == 'Daily Mail').mean()
print(f"Daily Mail Accuracy: {accuracy*100:.2f}%")
print(len(daily_mail_headlines_df))

## **Guardian Scraper**

In [ ]:
#import scraping libraries
import requests
from bs4 import BeautifulSoup

# make a scraping function that parses the RSS, retrieves all titles, runs through predict_headline function, outputing a print and overall accuracy
def scrape_and_predict(url, publisher):
    response = requests.get(url)
    xml_content = response.content

    soup = BeautifulSoup(xml_content, 'xml')

    items = soup.find_all('item')

    headlines = []

    # for each item, retrieve headline and add to headlines list
    for item in items:
        headline = item.title.text
        headlines.append(headline)

    # intiate a dataframe to store headlines
    df = pd.DataFrame(headlines, columns=['Headline'])
    for headline in headlines:
        # pass through predict_headline function
        company, confidence = predict_headline(headline)
        print(f"Prediction for '{headline}': {company}")
        # add company name to dataframe
        df.loc[df['Headline'] == headline, 'Predicted Publication'] = company

    # calculate publisher prediction accuracy
    accuracy = (df['Predicted Publication'] == publisher).mean()
    print(f"{publisher} Accuracy: {accuracy*100:.2f}%")
    print(f"Number of Headlines: {len(df)}")

In [ ]:
scrape_and_predict("https://www.theguardian.com/uk/rss", "The Guardian")

## **New York Times Scraper**

In [ ]:
scrape_and_predict("https://rss.nytimes.com/services/xml/rss/nyt/HomePage.xml", "New York Times")

### **New York Post Scraper**

In [ ]:
scrape_and_predict("https://nypost.com/feed/", "New York Post")

### **FOX News Scraper**

In [ ]:
scrape_and_predict("https://moxie.foxnews.com/google-publisher/latest.xml", "FOX")

## **CNBC Scraper**

In [ ]:
scrape_and_predict("https://search.cnbc.com/rs/search/combinedcms/view.xml?partnerId=wrss01&id=100003114", "CNBC")

## **USA Today Scraper**

In [ ]:
import requests
from bs4 import BeautifulSoup

# request to the usa_today Website
url = "https://www.usatoday.com/"

response = requests.get(url)

soup = BeautifulSoup(response.text, 'html.parser')

# retrieve all links
links = soup.find_all('a')

usa_today_headlines = []

for link in links:
    # retrieve title attribute from the link
    headline = link.get('title')
    # Only add headlines if they exist and are not empty
    if headline and headline.strip():
        usa_today_headlines.append(headline.strip())

usa_today_headlines_df = pd.DataFrame(usa_today_headlines, columns=['Headline'])

usa_today_headlines_df['Predicted Publication'] = ''

# pass each headline into predict_headline
for headline in usa_today_headlines:
    company, confidence = predict_headline(headline)
    # print company and confidence
    print(f"Prediction for '{headline}': {company}")
    # save headline with company into a dataframe
    usa_today_headlines_df.loc[usa_today_headlines_df['Headline'] == headline, 'Predicted Publication'] = company

# count the accuracy
accuracy = (usa_today_headlines_df['Predicted Publication'] == 'USA Today').mean()
print(f"USA Today Accuracy: {accuracy*100:.2f}%")
print(f"Number of Headlines: {len(usa_today_headlines_df)}")